# Boat Detection for Palm Beach County Inlets

This notebook detects and counts recreational boats in images from stationary cameras at inlet locations.

**Approach:**
1. Define a polygon mask to isolate water regions (exclude beach, parking lots, vegetation)
2. Use YOLOv11 with SAHI (Slicing Aided Hyper Inference) for small object detection
3. Filter detections to keep only boats within the water mask
4. Evaluate accuracy against ground truth annotations

---
## Step 1: Install Required Packages

We need:
- **ultralytics**: YOLOv11 object detection model
- **sahi**: Slicing Aided Hyper Inference - improves detection of small objects by running detection on image slices

In [ ]:
# Install headless OpenCV first (avoids libGL error)
!pip install -q opencv-python-headless

# Install SAHI and YOLOv11
!pip install -q sahi ultralytics

---
## Step 2: Import Libraries

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

# Import OpenCV (cv2)
try:
    import cv2
    print(f"OpenCV version: {cv2.__version__}")
except ImportError:
    print("Installing OpenCV...")
    import subprocess
    subprocess.run(['pip', 'install', 'opencv-python-headless'], check=True)
    import cv2
    print(f"OpenCV version: {cv2.__version__}")

# Import PyTorch
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Import SAHI
from sahi.predict import get_sliced_prediction
from sahi import AutoDetectionModel
print("SAHI imported successfully")

---
## Step 3: Configure Paths

Update these paths to match your file locations.

In [ ]:
# =============================================================================
# CONFIGURATION - UPDATE THESE PATHS
# =============================================================================

# Folder containing your inlet images
IMAGE_FOLDER = "/content/Jupiter_Inlet"

# Ground truth annotations in COCO format
GROUND_TRUTH_JSON = "/content/Jupiter_Inlet/_annotations.coco.json"

# Output folder for predictions
OUTPUT_FOLDER = "/content/output"

# Create output folder
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"Image folder: {IMAGE_FOLDER}")
print(f"Ground truth: {GROUND_TRUTH_JSON}")
print(f"Output folder: {OUTPUT_FOLDER}")

---
## Step 4: Define Water Polygon Mask

The water polygon defines which area of the image contains water. Boats detected outside this region (on beach, parking lot, etc.) will be filtered out.

**Important:** This polygon is specific to your camera position. If the camera moves or you use a different inlet, you'll need to redefine the polygon.

The polygon below is for the Jupiter Inlet camera (2000x1500 pixel images).

In [ ]:
# =============================================================================
# WATER POLYGON FOR JUPITER INLET
# =============================================================================
# This polygon traces the boundary between water and land.
# Points are (x, y) coordinates, listed in order around the water region.
# Image dimensions: 4352 x 3264 pixels

WATER_POLYGON = [
    (4352, 3264),   # Bottom-right corner (ocean)
    (2502, 3264),   # Bottom edge - where water meets beach
    (2393, 3046),   # Tracing waterline up the beach...
    (2284, 2828),
    (2132, 2611),
    (1958, 2393),
    (1784, 2176),
    (1632, 1958),
    (1479, 1740),
    (1349, 1523),
    (1218, 1305),   # Mid-left area near inlet
    (1088, 1196),
    (979, 1088),
    (913, 979),
    (870, 870),     # Inlet/jetty area
    (848, 761),
    (870, 696),
    (979, 652),     # Around the jetty
    (1196, 761),
    (1414, 826),
    (1632, 837),    # Right side of jetty
    (1523, 739),    # Distant shoreline
    (1349, 609),
    (1175, 478),
    (1000, 348),
    (870, 217),
    (761, 108),
    (696, 0),       # Top edge
    (4352, 0),      # Top-right corner (ocean/sky)
]

print(f"Water polygon has {len(WATER_POLYGON)} vertices")
print(f"Image dimensions expected: 4352 x 3264")

---
## Step 5: Create Water Mask Function

This function converts the polygon into a binary mask image where:
- White (255) = water (detect boats here)
- Black (0) = land (ignore detections here)

In [ ]:
def create_water_mask(image_path, polygon_points):
    """
    Create a binary mask from polygon points.
    
    Args:
        image_path: Path to an image (to get dimensions)
        polygon_points: List of (x, y) tuples defining water boundary
    
    Returns:
        mask: Binary numpy array (255=water, 0=land)
    """
    # Load image to get dimensions
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {image_path}")
    
    height, width = img.shape[:2]
    
    # Create empty mask (all black)
    mask = np.zeros((height, width), dtype=np.uint8)
    
    # Convert polygon to numpy array
    polygon = np.array(polygon_points, dtype=np.int32)
    
    # Fill polygon with white (255)
    cv2.fillPoly(mask, [polygon], 255)
    
    return mask

---
## Step 6: Create and Visualize the Water Mask

Let's create the mask and verify it looks correct.

In [ ]:
# Find first image in folder to create mask
image_files = sorted([
    f for f in os.listdir(IMAGE_FOLDER) 
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

if not image_files:
    raise FileNotFoundError(f"No images found in {IMAGE_FOLDER}")

# Use first image as reference
reference_image = os.path.join(IMAGE_FOLDER, image_files[0])
print(f"Reference image: {reference_image}")

# Create mask
WATER_MASK = create_water_mask(reference_image, WATER_POLYGON)
print(f"Mask shape: {WATER_MASK.shape}")
print(f"Water coverage: {np.sum(WATER_MASK > 0) / WATER_MASK.size * 100:.1f}% of image")

In [ ]:
# Visualize the mask
img = cv2.imread(reference_image)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Create masked version (water only)
masked_img = cv2.bitwise_and(img_rgb, img_rgb, mask=WATER_MASK)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img_rgb)
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(WATER_MASK, cmap='Blues')
axes[1].set_title('Water Mask (white = water)')
axes[1].axis('off')

axes[2].imshow(masked_img)
axes[2].set_title('Water Region Only')
axes[2].axis('off')

plt.tight_layout()
plt.show()

---
## Step 7: Define Detection Helper Function

This function checks if a detected bounding box is within the water region.

In [ ]:
def is_in_water(bbox, mask, min_overlap=0.3):
    """
    Check if a detection overlaps sufficiently with the water mask.
    
    We use overlap ratio instead of just checking the center point.
    This is more robust for boats partially visible at the water's edge.
    
    Args:
        bbox: Bounding box as (x, y, width, height)
        mask: Binary water mask
        min_overlap: Minimum fraction of bbox that must be in water (0.3 = 30%)
    
    Returns:
        True if detection is in water, False otherwise
    """
    x, y, w, h = [int(v) for v in bbox]
    
    # Clamp coordinates to image bounds
    x1 = max(0, x)
    y1 = max(0, y)
    x2 = min(mask.shape[1], x + w)
    y2 = min(mask.shape[0], y + h)
    
    # Check for valid box
    if x2 <= x1 or y2 <= y1:
        return False
    
    # Extract the mask region under the bounding box
    bbox_region = mask[y1:y2, x1:x2]
    
    # Calculate what fraction is water
    overlap_ratio = np.sum(bbox_region > 0) / bbox_region.size
    
    return overlap_ratio >= min_overlap

---
## Step 8: Configure Detection Parameters

These parameters control the detection accuracy:

- **MODEL_PATH**: Which YOLO model to use. Larger models are more accurate but slower.
- **CONFIDENCE_THRESHOLD**: Minimum confidence to consider a detection. Lower = more detections (but more false positives).
- **SLICE_SIZE**: Size of image slices for SAHI. Should be 3-5x the typical boat size in pixels.
- **OVERLAP_RATIO**: How much slices overlap. Higher = fewer missed detections at boundaries.

In [ ]:
# =============================================================================
# DETECTION PARAMETERS
# =============================================================================

# Model selection (larger = more accurate, slower)
# Options: yolo11n.pt, yolo11s.pt, yolo11m.pt, yolo11l.pt, yolo11x.pt
MODEL_PATH = "yolo11m.pt"

# Confidence threshold (0.0 to 1.0)
# Lower value catches more boats but may include false positives
# The water mask will filter out many false positives, so we can use a lower threshold
CONFIDENCE_THRESHOLD = 0.25

# Slice size for SAHI
# Typical boats are 50-150 pixels, so slice size of 384-512 works well
SLICE_SIZE = 512

# Overlap between slices (0.0 to 1.0)
# Higher overlap reduces missed detections at slice boundaries
OVERLAP_RATIO = 0.25

# IoU threshold for Non-Maximum Suppression
# Removes duplicate detections of the same boat
IOU_THRESHOLD = 0.5

print("Detection parameters:")
print(f"  Model: {MODEL_PATH}")
print(f"  Confidence threshold: {CONFIDENCE_THRESHOLD}")
print(f"  Slice size: {SLICE_SIZE}x{SLICE_SIZE}")
print(f"  Overlap ratio: {OVERLAP_RATIO}")

---
## Step 9: Load the Detection Model

We use SAHI's AutoDetectionModel wrapper around YOLOv11.

In [ ]:
# Determine device (GPU if available)
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load model
print(f"Loading model: {MODEL_PATH}...")
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=MODEL_PATH,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    device=device
)
print("Model loaded successfully!")

---
## Step 10: Load Ground Truth Annotations

Load the COCO-format annotations to map filenames to image IDs.

In [ ]:
# Load ground truth
with open(GROUND_TRUTH_JSON) as f:
    gt_data = json.load(f)

# Create mapping from filename to image ID
filename_to_id = {img['file_name']: img['id'] for img in gt_data['images']}

# Count ground truth boats per image
gt_counts = {}
for ann in gt_data['annotations']:
    img_id = ann['image_id']
    gt_counts[img_id] = gt_counts.get(img_id, 0) + 1

print(f"Ground truth loaded:")
print(f"  {len(gt_data['images'])} images")
print(f"  {len(gt_data['annotations'])} boat annotations")
print(f"  Average boats per image: {len(gt_data['annotations'])/len(gt_data['images']):.1f}")

---
## Step 11: Run Boat Detection

This is the main detection loop. For each image:
1. Run SAHI sliced prediction to detect objects
2. Filter for boat/ship class only
3. Filter to keep only detections within the water mask
4. Save predictions

In [ ]:
# Storage for predictions
all_predictions = []

# Statistics
stats = {
    'total_detections': 0,
    'boats_in_water': 0,
    'filtered_out': 0
}

# Process each image
print(f"\nProcessing {len(image_files)} images...\n")

for i, filename in enumerate(image_files):
    image_path = os.path.join(IMAGE_FOLDER, filename)
    
    # Run SAHI sliced prediction
    result = get_sliced_prediction(
        image=image_path,
        detection_model=detection_model,
        slice_height=SLICE_SIZE,
        slice_width=SLICE_SIZE,
        overlap_height_ratio=OVERLAP_RATIO,
        overlap_width_ratio=OVERLAP_RATIO,
        postprocess_type="NMS",
        postprocess_match_threshold=IOU_THRESHOLD,
    )
    
    # Filter detections
    image_boats = 0
    
    for obj in result.object_prediction_list:
        # Check if detection is a boat (COCO classes: 'boat', 'ship')
        if obj.category.name.lower() in ['boat', 'ship']:
            stats['total_detections'] += 1
            
            # Get bounding box in (x, y, width, height) format
            bbox = obj.bbox.to_xywh()
            
            # Check if boat is in water
            if is_in_water(bbox, WATER_MASK, min_overlap=0.3):
                stats['boats_in_water'] += 1
                image_boats += 1
                
                # Store prediction
                all_predictions.append({
                    "image_id": filename_to_id.get(filename, 0),
                    "category_id": 1,  # boat
                    "bbox": [float(b) for b in bbox],
                    "score": float(obj.score.value),
                    "filename": filename
                })
            else:
                stats['filtered_out'] += 1
    
    # Progress update
    gt_boats = gt_counts.get(filename_to_id.get(filename, -1), 0)
    print(f"[{i+1}/{len(image_files)}] {filename}: {image_boats} detected (GT: {gt_boats})")

print(f"\n" + "="*50)
print("DETECTION COMPLETE")
print("="*50)
print(f"Total boat detections: {stats['total_detections']}")
print(f"Boats in water (kept): {stats['boats_in_water']}")
print(f"Filtered out (on land): {stats['filtered_out']}")

---
## Step 12: Save Predictions

Save predictions in COCO format for evaluation.

In [ ]:
# Save predictions to JSON
predictions_path = os.path.join(OUTPUT_FOLDER, "predictions.json")

with open(predictions_path, "w") as f:
    json.dump(all_predictions, f, indent=2)

print(f"Predictions saved to: {predictions_path}")
print(f"Total predictions: {len(all_predictions)}")

---
## Step 13: Evaluate Accuracy

Calculate precision, recall, and F1 score by comparing predictions to ground truth.

- **Precision**: Of all detected boats, what fraction are real boats?
- **Recall**: Of all real boats, what fraction did we detect?
- **F1 Score**: Harmonic mean of precision and recall

In [ ]:
def calculate_iou(box1, box2):
    """
    Calculate Intersection over Union (IoU) between two boxes.
    Boxes are in (x, y, width, height) format.
    """
    # Convert values to float (in case they're strings from JSON)
    box1 = [float(v) for v in box1]
    box2 = [float(v) for v in box2]
    
    # Convert to (x1, y1, x2, y2) format
    b1_x1, b1_y1 = box1[0], box1[1]
    b1_x2, b1_y2 = box1[0] + box1[2], box1[1] + box1[3]
    
    b2_x1, b2_y1 = box2[0], box2[1]
    b2_x2, b2_y2 = box2[0] + box2[2], box2[1] + box2[3]
    
    # Calculate intersection
    inter_x1 = max(b1_x1, b2_x1)
    inter_y1 = max(b1_y1, b2_y1)
    inter_x2 = min(b1_x2, b2_x2)
    inter_y2 = min(b1_y2, b2_y2)
    
    if inter_x2 <= inter_x1 or inter_y2 <= inter_y1:
        return 0.0
    
    inter_area = (inter_x2 - inter_x1) * (inter_y2 - inter_y1)
    
    # Calculate union
    b1_area = box1[2] * box1[3]
    b2_area = box2[2] * box2[3]
    union_area = b1_area + b2_area - inter_area
    
    return inter_area / union_area if union_area > 0 else 0.0


def evaluate_predictions(gt_json, predictions, iou_threshold=0.5):
    """
    Calculate precision, recall, and F1 score.
    
    A prediction is a True Positive if it has IoU >= threshold with a ground truth box.
    Each ground truth box can only be matched once.
    """
    # Load ground truth
    with open(gt_json) as f:
        gt_data = json.load(f)
    
    # Organize ground truth by image
    gt_by_image = {}
    for ann in gt_data['annotations']:
        img_id = ann['image_id']
        if img_id not in gt_by_image:
            gt_by_image[img_id] = []
        gt_by_image[img_id].append(ann['bbox'])
    
    # Organize predictions by image
    pred_by_image = {}
    for pred in predictions:
        img_id = pred['image_id']
        if img_id not in pred_by_image:
            pred_by_image[img_id] = []
        pred_by_image[img_id].append((pred['bbox'], pred['score']))
    
    # Calculate metrics
    tp = 0  # True Positives
    fp = 0  # False Positives
    fn = 0  # False Negatives
    
    all_image_ids = set(list(gt_by_image.keys()) + list(pred_by_image.keys()))
    
    for img_id in all_image_ids:
        gt_boxes = gt_by_image.get(img_id, [])
        pred_boxes = pred_by_image.get(img_id, [])
        
        # Sort predictions by confidence (highest first)
        pred_boxes = sorted(pred_boxes, key=lambda x: x[1], reverse=True)
        
        # Track which ground truth boxes have been matched
        matched_gt = set()
        
        for pred_box, score in pred_boxes:
            best_iou = 0
            best_gt_idx = -1
            
            # Find best matching ground truth box
            for gt_idx, gt_box in enumerate(gt_boxes):
                if gt_idx in matched_gt:
                    continue
                iou = calculate_iou(pred_box, gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx
            
            # Check if match is good enough
            if best_iou >= iou_threshold:
                tp += 1
                matched_gt.add(best_gt_idx)
            else:
                fp += 1
        
        # Unmatched ground truth boxes are false negatives
        fn += len(gt_boxes) - len(matched_gt)
    
    # Calculate metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'fn': fn
    }

In [ ]:
# Run evaluation
metrics = evaluate_predictions(GROUND_TRUTH_JSON, all_predictions, iou_threshold=0.5)

print("\n" + "="*50)
print("EVALUATION RESULTS")
print("="*50)
print(f"\nTrue Positives (correct detections):  {metrics['tp']}")
print(f"False Positives (incorrect detections): {metrics['fp']}")
print(f"False Negatives (missed boats):        {metrics['fn']}")
print(f"\nPrecision: {metrics['precision']:.3f}  (of detections, how many are correct)")
print(f"Recall:    {metrics['recall']:.3f}  (of real boats, how many we found)")
print(f"F1 Score:  {metrics['f1']:.3f}  (overall accuracy)")

---
## Step 14: Visualize Results

Show detections vs ground truth for each image.

In [ ]:
def visualize_detections(image_path, gt_boxes, pred_boxes, water_mask):
    """
    Draw ground truth and predictions on image.
    
    Green boxes = Ground truth
    Red boxes = Predictions
    """
    # Load image
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Light blue overlay for water mask
    overlay = np.zeros_like(img)
    overlay[:, :, 2] = water_mask // 4  # Light blue tint
    img = cv2.addWeighted(img, 1.0, overlay, 0.2, 0)
    
    # Draw ground truth boxes (GREEN)
    for bbox in gt_boxes:
        x, y, w, h = [int(v) for v in bbox]
        cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 3)
    
    # Draw prediction boxes (RED)
    for bbox, score in pred_boxes:
        x, y, w, h = [int(v) for v in bbox]
        cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(img, f'{score:.2f}', (x, y-5), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    
    # Add legend
    cv2.rectangle(img, (10, 10), (250, 70), (255, 255, 255), -1)
    cv2.putText(img, f'GREEN = Ground Truth ({len(gt_boxes)})', (15, 35),
               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 180, 0), 2)
    cv2.putText(img, f'RED = Predictions ({len(pred_boxes)})', (15, 60),
               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180, 0, 0), 2)
    
    return img

In [ ]:
# Organize data for visualization
gt_by_image = {}
for ann in gt_data['annotations']:
    img_id = ann['image_id']
    if img_id not in gt_by_image:
        gt_by_image[img_id] = []
    gt_by_image[img_id].append(ann['bbox'])

pred_by_image = {}
for pred in all_predictions:
    img_id = pred['image_id']
    if img_id not in pred_by_image:
        pred_by_image[img_id] = []
    pred_by_image[img_id].append((pred['bbox'], pred['score']))

# Visualize each image
for filename in image_files:
    image_path = os.path.join(IMAGE_FOLDER, filename)
    img_id = filename_to_id.get(filename, -1)
    
    gt_boxes = gt_by_image.get(img_id, [])
    pred_boxes = pred_by_image.get(img_id, [])
    
    # Create visualization
    viz = visualize_detections(image_path, gt_boxes, pred_boxes, WATER_MASK)
    
    # Display
    plt.figure(figsize=(16, 12))
    plt.imshow(viz)
    plt.axis('off')
    plt.title(f'{filename}\nGround Truth: {len(gt_boxes)} boats | Predictions: {len(pred_boxes)} boats',
             fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"{filename}: GT={len(gt_boxes)}, Pred={len(pred_boxes)}")
    print("-" * 40)

---
## Step 15: Summary and Next Steps

### What we did:
1. Defined a water polygon mask to isolate the detection region
2. Used YOLOv11 with SAHI for small object detection
3. Filtered detections to keep only boats in water
4. Evaluated accuracy against ground truth

### To improve accuracy further:

**1. Fine-tune the model on your data:**
Pre-trained YOLO was not trained on aerial/elevated boat views. Fine-tuning on your annotated images will significantly improve accuracy.

**2. Adjust parameters:**
- Try different slice sizes (384, 512, 640)
- Adjust confidence threshold
- Try larger models (yolo11l.pt, yolo11x.pt)

**3. Create masks for other inlets:**
Each camera position needs its own water polygon.

**4. Implement tracking:**
To count unique boats crossing the inlet (not just detections per frame), implement object tracking across consecutive frames.

---
## Appendix: Interactive Polygon Creator

Use this tool to create water polygons for your other inlet cameras.

In [ ]:
def create_polygon_interactive(image_path):
    """
    Interactive tool to create a water polygon.
    
    Click on the image to add points around the water boundary.
    Points will be printed so you can copy them into WATER_POLYGON.
    """
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    points = []
    
    fig, ax = plt.subplots(figsize=(16, 12))
    ax.imshow(img_rgb)
    ax.set_title('Click to add points around the water boundary\n'
                'Start at one corner, go around the water region')
    
    def onclick(event):
        if event.xdata is not None and event.ydata is not None:
            x, y = int(event.xdata), int(event.ydata)
            points.append((x, y))
            
            # Draw point
            ax.plot(x, y, 'ro', markersize=8)
            ax.annotate(str(len(points)), (x+10, y), color='yellow', fontsize=10, fontweight='bold')
            
            # Draw line to previous point
            if len(points) > 1:
                ax.plot([points[-2][0], points[-1][0]], 
                       [points[-2][1], points[-1][1]], 'r-', linewidth=2)
            
            fig.canvas.draw()
            print(f"Point {len(points)}: ({x}, {y})")
    
    fig.canvas.mpl_connect('button_press_event', onclick)
    plt.show()
    
    # Print final polygon
    print("\n" + "="*50)
    print("COPY THIS POLYGON TO YOUR CODE:")
    print("="*50)
    print("WATER_POLYGON = [")
    for pt in points:
        print(f"    ({pt[0]}, {pt[1]}),")
    print("]")
    
    return points

# Uncomment to use:
# points = create_polygon_interactive("/path/to/your/image.jpg")